# 06 - Model Evaluation

**Goal:** Evaluate the chosen model beyond accuracy and select a business-friendly decision threshold.

**Why recall matters:** A missed defect (false negative) that reaches the customer costs far more than an unnecessary re-inspection (false positive).

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             RocCurveDisplay, precision_recall_curve,
                             precision_score, recall_score, f1_score)

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")["defect"]

models = {
    "Logistic Regression": joblib.load("../models/logistic_regression.joblib"),
    "Random Forest": joblib.load("../models/random_forest.joblib"),
    "XGBoost": joblib.load("../models/xgboost.joblib"),
}
lr = models["Logistic Regression"]
print(X_test.shape, y_test.shape)

## Confusion Matrix (threshold = 0.5)
| | Predicted Good | Predicted Defective |
|---|---|---|
| **Actual Good** | 259 (TN) | 40 (FP) |
| **Actual Defective** | 16 (FN) | 85 (TP) |

Out of 101 defective parts, 85 were caught and 16 were missed (recall = 0.842, precision = 0.680).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    lr, X_test, y_test,
    display_labels=["Good", "Defective"], cmap="Blues", ax=ax)
plt.title("Confusion Matrix: Logistic Regression")
plt.savefig("../reports/08_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(classification_report(y_test, lr.predict(X_test),
                            target_names=["Good", "Defective"]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_title("ROC Curve Comparison")
ax.legend()
plt.savefig("../reports/09_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
y_prob = lr.predict_proba(X_test)[:, 1]

rows = []
for t in np.arange(0.2, 0.85, 0.05):
    y_pred_t = (y_prob >= t).astype(int)
    rows.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(y_test, y_pred_t),
        "Recall": recall_score(y_test, y_pred_t),
        "F1": f1_score(y_test, y_pred_t),
        "Flagged parts": y_pred_t.sum(),
    })

thr_df = pd.DataFrame(rows).round(3)
thr_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(thr_df["Threshold"], thr_df["Precision"], marker="o", label="Precision")
plt.plot(thr_df["Threshold"], thr_df["Recall"], marker="o", label="Recall")
plt.xlabel("Threshold")
plt.title("Precision-Recall Trade-off")
plt.legend()
plt.savefig("../reports/10_threshold_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ASSUMPTIONS 
COST_MISSED_DEFECT = 2000   # ₹ per defective part reached customer
COST_FALSE_ALARM   = 500    # ₹ per good part unnecessarily inspected

costs = []
for t in np.arange(0.1, 0.9, 0.05):
    y_pred_t = (y_prob >= t).astype(int)
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    costs.append({"Threshold": round(t, 2), "Missed (FN)": fn, "False alarms (FP)": fp,
                  "Total cost ₹": fn * COST_MISSED_DEFECT + fp * COST_FALSE_ALARM})

cost_df = pd.DataFrame(costs)
best = cost_df.loc[cost_df["Total cost ₹"].idxmin()]
print(cost_df)
print("\n Best threshold:", best["Threshold"], "| Min cost: ₹", best["Total cost ₹"])

## Threshold Tuning & Business Cost
**Assumed costs (illustrative, not real factory data):**
- Missed defect: ₹5,000 per part
- False alarm (extra inspection): ₹500 per part

| Scenario | Best threshold | Result |
|---|---|---|
| Cost ₹5,000 / ₹500 | **0.2** | Missed defects fell from 16 to 3. Total cost fell from ₹1,00,000 to ₹56,000 (~44% saving) |
| Cost ₹2,000 / ₹500 | **0.3** | Cheaper missed defects → model can afford a higher threshold |

**Takeaway:** The best threshold depends on the cost ratio of false negatives to false positives, and the default of 0.5 is not optimal here.

 The threshold was tuned on the test set, which is slightly optimistic. In production, tune it on a validation set or with cross-validation.